# Forecast audit: validation selection, held-out error and forecast decomposition

All data is synthetic and educational.

In [1]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from database.database import read_table
from src.utils import inr

In [2]:
evaluation = read_table("forecast_metrics")
evaluation

,model,split,mae,rmse,wape,mape,bias_inr,bias_pct,zero_actual_months,selected
0,MA3 + 0% pipeline,test,1.590430e+08,1.834834e+08,0.138260,0.141106,-1.590430e+08,-0.138260,0,0
1,MA3 + 0% pipeline,validation,9.807762e+07,1.138232e+08,0.115968,0.118012,9.807762e+07,0.115968,0,0
2,MA3 + 25% pipeline,test,5.800023e+07,7.599079e+07,0.050421,0.050045,-6.430539e+06,-0.005590,0,0
3,MA3 + 25% pipeline,validation,1.804250e+08,1.865614e+08,0.213336,0.216403,1.804250e+08,0.213336,0,0
4,MA3 + 50% pipeline,test,1.461819e+08,1.582257e+08,0.127080,0.127105,1.461819e+08,0.127080,0,0
5,MA3 + 50% pipeline,validation,2.627723e+08,2.656001e+08,0.310705,0.314794,2.627723e+08,0.310705,0,0
6,SES + 0% pipeline,test,1.841995e+08,1.984078e+08,0.160129,0.158054,-1.841995e+08,-0.160129,0,0
7,SES + 0% pipeline,validation,1.004355e+08,1.134139e+08,0.118756,0.124449,9.689390e+07,0.114568,0,0
8,SES + 25% pipeline,test,5.257239e+07,6.372923e+07,0.045703,0.045217,-2.529792e+07,-0.021992,0,0
9,SES + 25% pipeline,validation,1.795372e+08,1.859356e+08,0.212286,0.218611,1.795372e+08,0.212286,0,0


In [3]:
backtest = read_table("backtest")
assert (
    backtest[backtest.split.eq("validation")].period.max()
    < backtest[backtest.split.eq("test")].period.min()
)
f = read_table("forecast")
print("Next three months:", inr(f.head(3).forecast.sum()))
f

Next three months: ₹2,76,42,07,292


,period,forecast,lower,upper,historical_baseline,pipeline_contribution,weighted_pipeline_component,baseline_component,closed_won_contribution,model,confidence_indicator,target,gap_to_target,expected_attainment,forecast_variance
0,2026-09-01,1.013892e+09,9.108507e+08,1.116933e+09,1.013892e+09,1.593476e+09,0.0,1.013892e+09,0.0,Seasonal naive + 0% pipeline,"Indicative range; 6 calibration origins, uncal...",1.192324e+09,1.784323e+08,0.850349,None
1,2026-10-01,8.610629e+08,7.153409e+08,1.006785e+09,8.610629e+08,9.995442e+08,0.0,8.610629e+08,0.0,Seasonal naive + 0% pipeline,"Indicative range; 6 calibration origins, uncal...",1.192324e+09,3.312611e+08,0.722172,None
2,2026-11-01,8.892527e+08,7.107805e+08,1.067725e+09,8.892527e+08,8.740546e+08,0.0,8.892527e+08,0.0,Seasonal naive + 0% pipeline,"Indicative range; 6 calibration origins, uncal...",1.192324e+09,3.030712e+08,0.745815,None
3,2026-12-01,7.691531e+08,5.630711e+08,9.752351e+08,7.691531e+08,5.117006e+08,0.0,7.691531e+08,0.0,Seasonal naive + 0% pipeline,"Indicative range; 6 calibration origins, uncal...",1.192324e+09,4.231709e+08,0.645087,None
4,2027-01-01,7.688446e+08,5.384379e+08,9.992513e+08,7.688446e+08,1.927337e+08,0.0,7.688446e+08,0.0,Seasonal naive + 0% pipeline,"Indicative range; 6 calibration origins, uncal...",1.192324e+09,4.234794e+08,0.644829,None
5,2027-02-01,7.721791e+08,5.197812e+08,1.024577e+09,7.721791e+08,6.105659e+07,0.0,7.721791e+08,0.0,Seasonal naive + 0% pipeline,"Indicative range; 6 calibration origins, uncal...",1.192324e+09,4.201449e+08,0.647625,None
6,2027-03-01,9.933323e+08,7.207115e+08,1.265953e+09,9.933323e+08,1.476853e+07,0.0,9.933323e+08,0.0,Seasonal naive + 0% pipeline,"Indicative range; 6 calibration origins, uncal...",1.192324e+09,1.989917e+08,0.833106,None
7,2027-04-01,9.636169e+08,6.721730e+08,1.255061e+09,9.636169e+08,3.720328e+06,0.0,9.636169e+08,0.0,Seasonal naive + 0% pipeline,"Indicative range; 6 calibration origins, uncal...",1.192324e+09,2.287071e+08,0.808184,None
8,2027-05-01,1.212284e+09,9.031609e+08,1.521407e+09,1.212284e+09,0.000000e+00,0.0,1.212284e+09,0.0,Seasonal naive + 0% pipeline,"Indicative range; 6 calibration origins, uncal...",1.192324e+09,-1.995993e+07,1.016740,None
9,2027-06-01,1.219654e+09,8.938096e+08,1.545498e+09,1.219654e+09,0.000000e+00,0.0,1.219654e+09,0.0,Seasonal naive + 0% pipeline,"Indicative range; 6 calibration origins, uncal...",1.192324e+09,-2.732984e+07,1.022921,None


## Interpretation

One-step backtest results cannot validate twelve-month performance. The displayed future range is indicative, not a guaranteed confidence interval.